### Importación de librerías

In [1]:
import pandas as pd
import numpy as np
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment
import os

# Archivo registros de Empadronamiento

## Inspeccionando Archivos

In [2]:
xl = pd.ExcelFile('entrada\CLIENTES ANSATTV.xlsx')

In [3]:
xl.sheet_names

['CLIENTES NOROCCIDENTE', 'CLIENTES SAN ANTONIO']

El archivo contiene 2 pestañas

In [4]:
df_preview = pd.read_excel(xl, nrows=1)
df_preview

,CLIENTES NOROCCIDENTE,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
0,CI,CONTRATO,FECHA,Nombre del Usuario,PLAN,TELEFONO,DIRECCION,IP


In [5]:
df_noroccidente = pd.read_excel(xl, sheet_name='CLIENTES NOROCCIDENTE', header=1)
#df_noroccidente.head(2)

In [6]:
df_noroccidente.shape

(487, 8)

El archivo $CLIENTES ANSATTV.xlsx$ contiene 487 registros en la primera hoja

In [7]:
df_noroccidente.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CI                  487 non-null    object
 1   CONTRATO            487 non-null    object
 2   FECHA               487 non-null    object
 3   Nombre del Usuario  487 non-null    object
 4   PLAN                487 non-null    object
 5   TELEFONO            487 non-null    object
 6   DIRECCION           486 non-null    object
 7   IP                  487 non-null    object
dtypes: object(8)
memory usage: 30.6+ KB


In [8]:
df_preview = pd.read_excel(xl, nrows=3, sheet_name='CLIENTES SAN ANTONIO')
#df_preview

In [9]:
df_san_antonio = pd.read_excel(xl, sheet_name='CLIENTES SAN ANTONIO', header=1)
#df_san_antonio.head(2)

El archivo $CLIENTES ANSATTV.xlsx$ contiene 448 registros en la segunda pestaña 

In [10]:
df_san_antonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CI/RUC     448 non-null    int64         
 1   #CONTRATO  448 non-null    int64         
 2   FECHA      448 non-null    datetime64[ns]
 3   NOMBRE     0 non-null      float64       
 4   PLAN       448 non-null    object        
 5   TELEFONO   448 non-null    object        
 6   DIRECCION  448 non-null    object        
 7   IP         448 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.1+ KB


## Registros sucursal Noroccidente

In [11]:
df_noroccidente.shape

(487, 8)

In [12]:
df_noroccidente.isna().sum()

CI                    0
CONTRATO              0
FECHA                 0
Nombre del Usuario    0
PLAN                  0
TELEFONO              0
DIRECCION             1
IP                    0
dtype: int64

In [13]:
np.where(df_noroccidente.isna())

(array([216]), array([6]))

In [14]:
#df_noroccidente.iloc[216]

In [15]:
conteo_contratos = df_noroccidente['CONTRATO'].value_counts()

In [16]:
contratos_repetidos = conteo_contratos[conteo_contratos > 1]

In [17]:
contratos_repetidos

CONTRATO
CON 0000919    2
Name: count, dtype: int64

In [18]:
#df_noroccidente[ df_noroccidente['CONTRATO'].str.contains('CON 0000919') ]

**Hallzgo**: Mismo contrato y cédula con direcciones IP **distintas**

In [19]:
contratos_duplicados = df_noroccidente[ df_noroccidente['CONTRATO'].str.contains('CON 0000919') ]

In [21]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', engine='openpyxl') as writer:
    contratos_duplicados.to_excel(writer, sheet_name='Duplicados_Contratos', index=False)

In [22]:
conteo_ci = df_noroccidente['CI'].value_counts()

In [23]:
ci_repeditos = conteo_ci[conteo_ci > 1]

In [24]:
#ci_repeditos

In [25]:
len(ci_repeditos)

21

In [26]:
ci_repeditos.sum()

np.int64(49)

In [27]:
#df_noroccidente[ df_noroccidente['CI'] == '']

In [28]:
#for ci, cantidad in ci_repeditos.items():
#    print (ci)

números de cédulas repetidas

In [29]:
lista_registros_ci = []
for ci, cantidad in ci_repeditos.items():
    df_ci = df_noroccidente[df_noroccidente['CI'] == ci]
    lista_registros_ci.append(df_ci)


In [30]:
duplicados_ci = pd.concat(lista_registros_ci, ignore_index=True)

In [31]:
duplicados_ci.shape

(49, 8)

In [32]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    duplicados_ci.to_excel(writer, sheet_name='Duplicados_CI', index=False)

In [33]:
pd.pivot_table(
    df_noroccidente,
    values='CONTRATO',
    index='PLAN',
    aggfunc='count'
)

,CONTRATO
PLAN,
250 MB,440
300 MB,34
350 MB,10
400 MB,1
450 MB,2


In [34]:
#duplicados_ci.head(3)

In [37]:
pivot_completo = (duplicados_ci
    .groupby('CI')
    .agg(
        Contratos=('CONTRATO', 'count'),
        IPs=('IP', 'nunique'),
        Telefonos=('TELEFONO', 'nunique'),
        Nombres=('Nombre del Usuario', 'nunique')
    )
    .assign(
        Tipo_Duplicidad=lambda x: 
            np.where(x['Nombres'] > 1, 'Diferentes_nombres',
            np.where((x['IPs'] > 1) & (x['Telefonos'] == 1), 'Mismo_nombre_diferentes_IP',
            np.where((x['IPs'] == 1) & (x['Telefonos'] == 1), 'Registro_duplicado_identico',
            np.where((x['IPs'] > 1) & (x['Telefonos'] > 1), 'Multiples_IP_y_telefonos', 'Patron_mixto'))))
    )
    .sort_values('Contratos', ascending=False)
)

#print(pivot_completo)

In [38]:
with pd.ExcelWriter('Proc_noroccidente.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    pivot_completo.to_excel(writer, sheet_name='Duplicidad_CI', index=True)

In [40]:
#df_noroccidente[ df_noroccidente['CI'] == '' ]

In [41]:
listado_ip = df_noroccidente['IP'].value_counts()

In [42]:
duplicados_ip = listado_ip[ listado_ip > 1 ]

In [43]:
duplicados_ip

Series([], Name: count, dtype: int64)

## Registros sucursal San Antonio

In [44]:
df_san_antonio.shape

(448, 8)

In [45]:
df_san_antonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CI/RUC     448 non-null    int64         
 1   #CONTRATO  448 non-null    int64         
 2   FECHA      448 non-null    datetime64[ns]
 3   NOMBRE     0 non-null      float64       
 4   PLAN       448 non-null    object        
 5   TELEFONO   448 non-null    object        
 6   DIRECCION  448 non-null    object        
 7   IP         448 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.1+ KB


In [46]:
df_san_antonio.isna().sum()

CI/RUC         0
#CONTRATO      0
FECHA          0
NOMBRE       448
PLAN           0
TELEFONO       0
DIRECCION      0
IP             0
dtype: int64

In [47]:
conteo_contratos_sa = df_san_antonio['#CONTRATO'].value_counts()

In [48]:
contratos_repetidos_sa = conteo_contratos_sa[conteo_contratos_sa > 1]

In [49]:
contratos_repetidos_sa

#CONTRATO
10397    2
1359     2
1460     2
Name: count, dtype: int64

In [50]:
#df_san_antonio[ df_san_antonio['#CONTRATO'] == 10397 ]

**Hallazgo:** Mismo contrato y cédula con direcciones IP **distintas**

In [58]:
lista_contratos = []
for c, cantidad in contratos_repetidos_sa.items():
    df_c = df_san_antonio[df_san_antonio['#CONTRATO'] == c]
    lista_contratos.append(df_c)

In [59]:
duplicados_c = pd.concat(lista_contratos, ignore_index=True)

In [60]:
#duplicados_c

In [61]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', engine='openpyxl') as writer:
    duplicados_c.to_excel(writer, sheet_name='Duplicados_Contratos', index=False)

In [62]:
df_san_antonio.columns.values

array(['CI/RUC', '#CONTRATO', 'FECHA', 'NOMBRE', 'PLAN', 'TELEFONO',
       'DIRECCION', 'IP'], dtype=object)

In [63]:
conteo_ci_sa = df_san_antonio['CI/RUC'].value_counts()

In [64]:
ci_repeditos_sa = conteo_ci_sa[conteo_ci_sa > 1]

In [65]:
#ci_repeditos_sa

In [66]:
len(ci_repeditos_sa)

4

In [67]:
ci_repeditos_sa.sum()

np.int64(9)

**Hallazgo:** números de cédulas repetidas

In [71]:
lista_registros_ci_sa = []
for ci, cantidad in ci_repeditos_sa.items():
    df_ci = df_san_antonio[df_san_antonio['CI/RUC'] == ci]
    lista_registros_ci_sa.append(df_ci)


In [72]:
duplicados_ci_sa = pd.concat(lista_registros_ci_sa, ignore_index=True)

In [73]:
duplicados_ci_sa.shape

(9, 8)

In [74]:
#duplicados_ci_sa

In [75]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    duplicados_ci_sa.to_excel(writer, sheet_name='Duplicados_CI', index=False)

In [76]:
pd.pivot_table(
    df_san_antonio,
    values='#CONTRATO',
    index='PLAN',
    aggfunc='count'
)

,#CONTRATO
PLAN,
250Mbps,330
300Mbps,72
350Mbps,29
400Mbps,5
450Mbps,12


In [77]:
pivot_completo_sa = (duplicados_ci_sa
    .groupby('CI/RUC')
    .agg(
        Contratos=('#CONTRATO', 'count'),
        IPs=('IP', 'nunique'),
        Telefonos=('TELEFONO', 'nunique'),
        Nombres=('NOMBRE', 'nunique')
    )
    .assign(
        Tipo_Duplicidad=lambda x: 
            np.where(x['Nombres'] > 1, 'Diferentes_nombres',
            np.where((x['IPs'] > 1) & (x['Telefonos'] == 1), 'Mismo_nombre_diferentes_IP',
            np.where((x['IPs'] == 1) & (x['Telefonos'] == 1), 'Registro_duplicado_identico',
            np.where((x['IPs'] > 1) & (x['Telefonos'] > 1), 'Multiples_IP_y_telefonos', 'Patron_mixto'))))
    )
    .sort_values('Contratos', ascending=False)
)

#print(pivot_completo_sa)

In [78]:
with pd.ExcelWriter('Proc_san_antonio.xlsx', mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    pivot_completo_sa.to_excel(writer, sheet_name='Duplicidad_CI', index=True)

In [79]:
listado_ip_sa = df_san_antonio['IP'].value_counts()

In [80]:
duplicados_ip_sa = listado_ip_sa[ listado_ip_sa > 1 ]

In [81]:
duplicados_ip_sa

Series([], Name: count, dtype: int64)

No se visualizan IPs duplicadas

## JOIN (inner, outer) noroccidental y san_antonio

Encontrar números de contratos en ambas tablas

In [83]:
df_noroccidente.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487 entries, 0 to 486
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   CI                  487 non-null    object
 1   CONTRATO            487 non-null    object
 2   FECHA               487 non-null    object
 3   Nombre del Usuario  487 non-null    object
 4   PLAN                487 non-null    object
 5   TELEFONO            487 non-null    object
 6   DIRECCION           486 non-null    object
 7   IP                  487 non-null    object
dtypes: object(8)
memory usage: 30.6+ KB


In [84]:
df_san_antonio.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 448 entries, 0 to 447
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   CI/RUC     448 non-null    int64         
 1   #CONTRATO  448 non-null    int64         
 2   FECHA      448 non-null    datetime64[ns]
 3   NOMBRE     0 non-null      float64       
 4   PLAN       448 non-null    object        
 5   TELEFONO   448 non-null    object        
 6   DIRECCION  448 non-null    object        
 7   IP         448 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(4)
memory usage: 28.1+ KB


Extracción de números dentro de la columna CONTRATO

In [85]:
df_noroccidente['CONTRATO_NUM'] = df_noroccidente['CONTRATO'].str.extract('(\d+)')

Casteo de númerico a string

In [86]:
df_noroccidente['CONTRATO_NUM'] = df_noroccidente['CONTRATO_NUM'].astype(str)

In [87]:
df_noroccidente['CONTRATO_NUM'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 487 entries, 0 to 486
Series name: CONTRATO_NUM
Non-Null Count  Dtype 
--------------  ----- 
487 non-null    object
dtypes: object(1)
memory usage: 3.9+ KB


In [88]:
df_noroccidente['CONTRATO_NUM'].iloc[:5]

0    0000534
1    0000537
2    0000538
3    0000545
4    0000568
Name: CONTRATO_NUM, dtype: object

In [89]:
df_san_antonio['CONTRATO_NUM'] = df_san_antonio['#CONTRATO'].astype(str)

In [90]:
df_san_antonio['CONTRATO_NUM'].info()

<class 'pandas.core.series.Series'>
RangeIndex: 448 entries, 0 to 447
Series name: CONTRATO_NUM
Non-Null Count  Dtype 
--------------  ----- 
448 non-null    object
dtypes: object(1)
memory usage: 3.6+ KB


**Verificación si las columnas tienen datos**

Valores Nulos

In [91]:
df_noroccidente.CONTRATO_NUM.isnull().sum()

np.int64(0)

Valores No nulos

In [92]:
df_noroccidente.CONTRATO_NUM.notna().sum()

np.int64(487)

In [93]:
df_noroccidente.CONTRATO_NUM.isna().sum()

np.int64(0)

In [94]:
df_san_antonio.CONTRATO_NUM.isnull().sum()

np.int64(0)

In [96]:
df_san_antonio.CONTRATO_NUM.notna().sum()

np.int64(448)

In [97]:
df_san_antonio.CONTRATO_NUM.isna().sum()

np.int64(0)

Comprobación de formatos y tipos de datos

In [98]:
df_noroccidente.CONTRATO_NUM.dtype, df_san_antonio.CONTRATO_NUM.dtype

(dtype('O'), dtype('O'))

**Normalización de datos**

Se eliminaran los ceros $0$ del lado izquierdo de cada número de contrato a demas de dejar solo los caracteres numéricos

In [99]:
df_noroccidente['CONTRATO_NORM'] = (
    df_noroccidente['CONTRATO_NUM']
    .astype(str)
    .str.strip() # Eliminar espacios
    .str.lstrip('0') # Eliminar ceros a la izquierda
)

In [100]:
df_noroccidente.iloc[:5, [1,8,9]]

,CONTRATO,CONTRATO_NUM,CONTRATO_NORM
0,CON 0000534,0000534,534
1,CON 0000537,0000537,537
2,CON 0000538,0000538,538
3,CON 0000545,0000545,545
4,CON 0000568,0000568,568


In [101]:
df_san_antonio['CONTRATO_NORM'] = (
    df_san_antonio['CONTRATO_NUM']
    .astype(str)
    .str.strip() # Eliminar espacios
    .str.lstrip('0') # Eliminar ceros a la izquierda
)

In [102]:
df_san_antonio.columns.get_indexer(['#CONTRATO', 'CONTRATO_NUM', 'CONTRATO_NORM'])

array([1, 8, 9])

In [103]:
df_san_antonio.iloc[ :5, [1,8,9] ]

,#CONTRATO,CONTRATO_NUM,CONTRATO_NORM
0,10391,10391,10391
1,125,125,125
2,123,123,123
3,149,149,149
4,45,45,45


In [104]:
df_san_antonio.CONTRATO_NORM.isin(['534']).sum()

np.int64(0)

In [105]:
np.where( df_san_antonio['CONTRATO_NORM'] == '534' ) [0]

array([], dtype=int64)

In [106]:
df_san_antonio.CONTRATO_NORM.str.contains('543').sum()

np.int64(0)

In [107]:
contratos_inner = pd.merge(
    df_noroccidente,
    df_san_antonio,
    how='inner',
    left_on='CONTRATO_NORM',
    right_on='CONTRATO_NORM',
    suffixes=('_norocc', '_san_anto')
)

Posibles contratos duplicados

In [108]:
contratos_inner.head()

,CI,CONTRATO,FECHA_norocc,Nombre del Usuario,PLAN_norocc,TELEFONO_norocc,DIRECCION_norocc,IP_norocc,CONTRATO_NUM_norocc,CONTRATO_NORM,CI/RUC,#CONTRATO,FECHA_san_anto,NOMBRE,PLAN_san_anto,TELEFONO_san_anto,DIRECCION_san_anto,IP_san_anto,CONTRATO_NUM_san_anto
0,1900101518,CON 0006005,2025-07-05 00:00:00,CORONEL ORTEGA MARIA MAGDALENA,250 MB,0979053110,NANEGAL,10.0.53.173,0006005,6005,1700609546,6005,2023-04-27,NaN,350Mbps,988776663,PARQUE CENTRAL DE POMASQUI,10.0.77.91,6005
1,1004151302,CON 0006040,2025-07-24 00:00:00,ANDRADE VALLE JENIFER CRISTINA,250 MB,0996622766,NANEGAL,10.0.53.201,0006040,6040,1715039804,6040,2023-09-29,NaN,250Mbps,961238338,CATEQUILLA,10.0.77.137,6040
2,1727385328,CON 0006052,2025-07-27 00:00:00,ANDRADE MORALES ERIKA FERNANDA,250 MB,0982175437,NANEGAL,10.0.66.157,0006052,6052,1717439226,6052,2024-01-26,NaN,250Mbps,986011631,TANLAHUA,10.0.77.179,6052
3,1704786480,CON 0006056,2025-07-31 00:00:00,AYALA REVELO RUTH GUADALUPE,250 MB,0995776130,NANEGAL,10.0.53.203,0006056,6056,1726473984,6056,2023-11-30,NaN,250Mbps,983609602,ATAHUALPA,10.0.77.162,6056
4,1004368468,CON 0006064,2025-08-13 00:00:00,BENALCAZAR JESSICA MARIELA,250 MB,0981781239,NANEGAL,10.0.53.205,0006064,6064,1726355413,6064,2023-12-27,NaN,250Mbps,939246066,JULIAN ARBAIZA Y VENEZUELA,10.0.77.171,6064


Como resultado se obtienen un total de 25 Contratos

In [109]:
print(f'Como resultado se obtienen un total de {contratos_inner.shape[0]} contratos posiblemente coincidentes en ambas sucursales')

Como resultado se obtienen un total de 25 contratos posiblemente coincidentes en ambas sucursales


In [110]:
with pd.ExcelWriter('Proc_Clientes.xlsx', engine='openpyxl') as writer:
    contratos_inner.to_excel(writer, sheet_name='inner', index=False)

In [111]:
with pd.ExcelWriter('Proc_Clientes.xlsx', engine='openpyxl', if_sheet_exists='replace', mode='a') as writer:
    contratos_inner.to_excel(writer, sheet_name='inner', index=False)

In [113]:
#os.getcwd()

In [114]:
contratos_outer = pd.merge(
    df_noroccidente,
    df_san_antonio,
    how='outer',
    left_on='CONTRATO_NORM',
    right_on='CONTRATO_NORM',
    suffixes=('_norocc', '_san_anto')
)

In [115]:
contratos_outer.shape

(910, 19)

In [117]:
with pd.ExcelWriter('Proc_Clientes.xlsx', mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    contratos_outer.to_excel(writer, sheet_name='outer', index=False)

In [118]:
#!pip install nbconvert[webpdf]
#!playwright install chromium

In [121]:
!jupyter nbconvert --to webpdf analisis_base_ansattv.ipynb

[NbConvertApp] Converting notebook analisis_base_ansattv.ipynb to webpdf
[NbConvertApp] Building PDF
[NbConvertApp] PDF successfully created
[NbConvertApp] Writing 438735 bytes to analisis_base_ansattv.pdf
